# Eksperimen Utama: modul temporal pada Multi-GIN+EU dan MEGA-GIN (110 run)

Tahap 5 rencana tesis. Notebook ini dijalankan **berulang kali** (satu sesi commit = maks. ~11 jam run); tiap sesi melanjutkan antrean dari run yang belum selesai.

**Rancangan (110 run):**

| Konfigurasi | Override `cfg.temporal` | Backbone | Seed | Gelombang |
|---|---|---|---|---|
| `base` | - | Multi-GIN, MEGA-GIN | 1-10 | 1 |
| `full` | Time2Vec + peluruhan | Multi-GIN, MEGA-GIN | 1-10 | 1 |
| `t2v` | Time2Vec | Multi-GIN | 1-5 | 2 |
| `sin` | sinusoidal tetap | Multi-GIN | 1-5 | 2 |
| `decay` | peluruhan | Multi-GIN | 1-5 | 2 |

x 2 dataset (HI-Small, LI-Small). Urutan antrean: gelombang 1 LI, gelombang 1 HI, gelombang 2 LI, gelombang 2 HI; di dalamnya per seed, sehingga pasangan baseline/lengkap untuk uji Wilcoxon terkumpul bertahap.

**Protokol** (`configs/thesis/`): `batch_accumulation 4`, HI maks. 20 epoch (patience 5), LI maks. 40 epoch (patience 8), test sekali di checkpoint val terbaik. **Dua run paralel**, satu per GPU T4.

**Setiap sesi:**
1. Settings: Input dataset IBM AML, **GPU T4 x2**, **Internet On**. Opsional (disarankan): **Add Input -> Output versi notebook ini sebelumnya** (lihat bagian 2) agar run yang sudah selesai dilewati otomatis dan hasil terkumpul di satu tempat.
2. Sesi pertama: biarkan `RUN_SMOKE = True` (uji singkat modul temporal ~15 menit). Sesi berikutnya: ubah ke `False`.
3. **Save Version -> Save & Run All (Commit)**.
4. Unduh `results_main.zip` dari Output, simpan ke `project_aml/hasil_kaggle/main/sesi_<N>/`.

In [ ]:
import os, time

NOTEBOOK_START = time.time()
SESSION_BUDGET_H = 11.0   # batas sesi commit Kaggle 12 jam; sisakan waktu untuk setup & zip
RUN_SMOKE = True          # sesi pertama True; sesi berikutnya False

## 1. Setup

In [ ]:
REPO_URL = "https://github.com/aanpujianto070990/MEGA-GNN-TMLR.git"
REPO_DIR = "/kaggle/working/MEGA-GNN-TMLR"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git fetch origin && git reset --hard origin/main

assert os.path.isdir(f"{REPO_DIR}/MegaGNN"), "Clone gagal -- pastikan Settings -> Internet = On, lalu jalankan ulang sel ini."
!git -C {REPO_DIR} log -1 --oneline

In [ ]:
import sys, torch

print("python", sys.version.split()[0], "| torch", torch.__version__, "| cuda", torch.version.cuda)
N_GPU = torch.cuda.device_count()
for i in range(N_GPU):
    print(f"gpu {i}:", torch.cuda.get_device_name(i))
if N_GPU < 2:
    print("PERINGATAN: hanya", N_GPU, "GPU -- pilih Accelerator 'GPU T4 x2'. Run akan dijalankan satu per satu.")

TORCH_MM = ".".join(torch.__version__.split("+")[0].split(".")[:2]) + ".0"
CUDA_TAG = ("cu" + torch.version.cuda.replace(".", "")) if torch.version.cuda else "cpu"
PYG_WHL = f"https://data.pyg.org/whl/torch-{TORCH_MM}+{CUDA_TAG}.html"

!pip install -q torch_geometric yacs ogb gdown tensorboardX torchmetrics pybind11
!pip install -q --only-binary=:all: pyg_lib torch_scatter torch_sparse -f {PYG_WHL}
!pip uninstall -y -q genagg
!pip install -q --no-deps {REPO_DIR}/genagg

!cd {REPO_DIR}/MegaGNN/cpp && python setup.py build_ext --inplace > build.log 2>&1; ls *.so
!cd {REPO_DIR} && python -c "import MegaGNN, torch_scatter, torch_sparse; from MegaGNN.cpp import ports_cpp; print('Import MegaGNN OK')"

## 2. Data, cache praproses, dan run yang sudah selesai

Run dianggap **selesai** jika `run_summary.json`-nya ditemukan di:
- `/kaggle/input/**` -- Output versi notebook ini sebelumnya yang ditambahkan sebagai Input (*Add Input -> Your Work -> Notebooks*). Folder run-nya disalin ke Output sesi ini (tanpa checkpoint), sehingga **Output versi terbaru selalu berisi seluruh hasil** dan sesi berikutnya cukup memakai versi terbaru sebagai Input;
- `thesis/done_runs.txt` di repo (cadangan, jika hasil disimpan di luar Kaggle).

In [ ]:
import glob, re, shutil

DATA_DIR = "/tmp/aml_data"
OUT_DIR = "/kaggle/working/results_main"
os.makedirs(OUT_DIR, exist_ok=True)

DATASETS = {"Small-HI": "HI-Small_Trans.csv", "Small-LI": "LI-Small_Trans.csv"}
for name, fname in DATASETS.items():
    matches = glob.glob(f"/kaggle/input/**/{fname}", recursive=True)
    assert matches, f"{fname} tidak ditemukan di /kaggle/input -- tambahkan dataset IBM AML sebagai Input"
    raw_dir = f"{DATA_DIR}/AML/{name}/raw"
    os.makedirs(raw_dir, exist_ok=True)
    if not os.path.exists(f"{raw_dir}/{fname}"):
        os.symlink(matches[0], f"{raw_dir}/{fname}")
    print(f"{name}: {matches[0]}")

RUN_DIR_RE = re.compile(r"AML-(Small-(?:HI|LI))-(Multi-GIN|MEGA-GIN)-(base|full|t2v|sin|decay|smoke-\w+)-gpu\d+$")


def run_key(summary_path):
    """(dataset, model, config, seed) dari .../AML-<ds>-<model>-<cfg>-gpu0/<seed>/run_summary.json"""
    seed_dir = os.path.dirname(summary_path)
    m = RUN_DIR_RE.search(os.path.basename(os.path.dirname(seed_dir)))
    return (m.group(1), m.group(2), m.group(3), int(os.path.basename(seed_dir))) if m else None


copied = 0
for path in glob.glob("/kaggle/input/**/run_summary.json", recursive=True):
    key = run_key(path)
    if key is None or key[2].startswith("smoke"):
        continue
    src = os.path.dirname(path)
    dst = f"{OUT_DIR}/AML-{key[0]}-{key[1]}-{key[2]}-gpu0/{key[3]}"
    if not os.path.exists(f"{dst}/run_summary.json"):
        shutil.copytree(src, dst, ignore=shutil.ignore_patterns("ckpt"), dirs_exist_ok=True)
        copied += 1
print(f"Run dari Input yang disalin ke Output sesi ini: {copied}")

DONE_FILE = set()
for line in open(f"{REPO_DIR}/thesis/done_runs.txt"):
    line = line.strip()
    if line and not line.startswith("#"):
        ds, model, cfg_, seed = line.split(",")
        DONE_FILE.add((ds, model, cfg_, int(seed)))
print(f"Run selesai menurut thesis/done_runs.txt: {len(DONE_FILE)}")

for name in DATASETS:
    !cd {REPO_DIR} && python thesis/prepare_data.py --dir {DATA_DIR} --name {name}

## 3. Rencana, penjadwal 2 GPU, dan helper

In [ ]:
import json, subprocess
import numpy as np
import pandas as pd

OVERRIDES = {
    "base": [],
    "t2v": ["temporal.encoding", "time2vec"],
    "sin": ["temporal.encoding", "sinusoid"],
    "decay": ["temporal.decay", "True"],
    "full": ["temporal.encoding", "time2vec", "temporal.decay", "True"],
}

# Perkiraan konservatif durasi satu run (jam) saat dua run berjalan paralel, epoch maksimum,
# termasuk +10% untuk modul temporal (dari pilot: HI ~311 s/epoch, LI ~448 s/epoch berurutan;
# paralel ~1,12x lebih lambat).
EST_H = {("Small-HI", "Multi-GIN"): 2.2, ("Small-HI", "MEGA-GIN"): 2.1,
         ("Small-LI", "Multi-GIN"): 6.2, ("Small-LI", "MEGA-GIN"): 6.0}


def build_plan():
    plan = []
    for ds in ["Small-LI", "Small-HI"]:                       # gelombang 1
        for seed in range(1, 11):
            for model in ["Multi-GIN", "MEGA-GIN"]:
                for cfg_ in ["base", "full"]:
                    plan.append((ds, model, cfg_, seed))
    for ds in ["Small-LI", "Small-HI"]:                       # gelombang 2 (ablasi)
        for seed in range(1, 6):
            for cfg_ in ["t2v", "sin", "decay"]:
                plan.append((ds, "Multi-GIN", cfg_, seed))
    return plan


PLAN = build_plan()
assert len(PLAN) == 110 and len(set(PLAN)) == 110


def run_dir_of(key):
    ds, model, cfg_, seed = key
    return f"{OUT_DIR}/AML-{ds}-{model}-{cfg_}-gpu0/{seed}"


def is_done(key):
    return key in DONE_FILE or os.path.exists(f"{run_dir_of(key)}/run_summary.json")


def launch(key, gpu, extra=()):
    ds, model, cfg_, seed = key
    cmd = ["python", "-u", "-m", "MegaGNN.main", "--cfg", f"configs/thesis/AML-{ds}-{model}.yaml",
           "--gpu", "0", "seed", str(seed), "out_dir", OUT_DIR, "dataset.dir", DATA_DIR,
           "name_tag", cfg_, *OVERRIDES.get(cfg_, []), *extra]
    os.makedirs(f"{OUT_DIR}/stdout", exist_ok=True)
    log = open(f"{OUT_DIR}/stdout/{ds}_{model}_{cfg_}_s{seed}.txt", "w")
    env = {**os.environ, "CUDA_VISIBLE_DEVICES": str(gpu)}   # tiap proses melihat satu GPU sebagai cuda:0
    proc = subprocess.Popen(cmd, cwd=REPO_DIR, env=env, stdout=log, stderr=subprocess.STDOUT)
    return proc, log


def last_status(key):
    path = f"{run_dir_of(key)}/logging.log"
    if not os.path.exists(path):
        return "(memuat data)"
    lines = [l.strip() for l in open(path) if l.startswith(("> Epoch", "Early stopping", "Best epoch"))]
    return lines[-1][:150] if lines else "(epoch pertama)"


def run_queue(queue, deadline, extra_for=lambda key: (), est=lambda key: EST_H[(key[0], key[1])]):
    """Jalankan antrean di semua GPU. Run hanya dimulai jika perkiraan durasinya selesai sebelum deadline."""
    gpus = list(range(max(1, N_GPU)))
    running, finished, failed = {}, [], []
    last_print = 0
    queue = list(queue)
    while queue or running:
        # bereskan run yang selesai
        for gpu, (key, proc, log, t0) in list(running.items()):
            if proc.poll() is not None:
                log.close()
                dur = (time.time() - t0) / 3600
                if proc.returncode == 0 and os.path.exists(f"{run_dir_of(key)}/run_summary.json"):
                    finished.append(key)
                    print(f"[selesai {dur:.2f} jam] {key}")
                else:
                    failed.append(key)
                    tail = "".join(open(log.name).readlines()[-30:])
                    print(f"[GAGAL exit {proc.returncode}] {key}\n{tail}")
                del running[gpu]
        # isi GPU kosong dengan run pertama di antrean yang masih sempat selesai
        for gpu in gpus:
            if gpu in running or not queue:
                continue
            left_h = (deadline - time.time()) / 3600
            fit = next((k for k in queue if est(k) <= left_h), None)
            if fit is None:
                continue
            queue.remove(fit)
            proc, log = launch(fit, gpu, extra_for(fit))
            running[gpu] = (fit, proc, log, time.time())
            print(f"[mulai gpu{gpu}] {fit} | perkiraan {est(fit):.1f} jam | sisa waktu sesi {left_h:.1f} jam")
            time.sleep(30)   # jeda kecil agar pemuatan data dua proses tidak bersamaan
        if not running and queue and all(est(k) > (deadline - time.time()) / 3600 for k in queue):
            print(f"Sisa waktu sesi tidak cukup untuk run berikutnya ({len(queue)} run tersisa di antrean).")
            break
        if time.time() - last_print > 900:
            last_print = time.time()
            print(f"--- {(time.time() - NOTEBOOK_START) / 3600:.2f} jam sejak notebook mulai")
            for gpu, (key, *_rest) in running.items():
                print(f"  gpu{gpu} {key}: {last_status(key)}")
        time.sleep(60)
    return finished, failed

## 4. Uji singkat modul temporal (sesi pertama saja)

Tiga run pendek di HI (1 epoch x 8 iterasi; evaluasi val/test tetap penuh) yang mencakup semua jalur kode baru: Multi-GIN `full` (Time2Vec + `DecayGINEConv`), Multi-GIN `sin` (pengodean beku), MEGA-GIN `full` (peluruhan pada agregasi sisi paralel). Jika ada yang gagal, notebook berhenti di sini agar kuota tidak terbuang.

In [ ]:
if RUN_SMOKE:
    SMOKE_EXTRA = ["optim.max_epoch", "1", "train.iter_per_epoch", "8"]
    smoke = [("Small-HI", "Multi-GIN", "smoke-full", 1), ("Small-HI", "Multi-GIN", "smoke-sin", 1),
             ("Small-HI", "MEGA-GIN", "smoke-full", 1)]
    OVERRIDES["smoke-full"] = OVERRIDES["full"]
    OVERRIDES["smoke-sin"] = OVERRIDES["sin"]
    deadline = NOTEBOOK_START + SESSION_BUDGET_H * 3600
    ok, bad = run_queue(smoke, deadline, extra_for=lambda k: SMOKE_EXTRA, est=lambda k: 0.3)
    for key in ok:
        s = json.load(open(f"{run_dir_of(key)}/run_summary.json"))
        t = s["temporal"]
        periods = t.get("time2vec_periods_days")
        print(f"{key}: params {s['params']} | temporal {t['encoding']}, decay {t['decay']} | "
              f"lambda/hari {t.get('lambda_per_day')} | periode T2V (hari) "
              f"{[round(p, 3) for p in periods] if periods else None} | test F1 {100 * s['test']['f1']:.2f}")
    assert not bad, f"Uji singkat gagal: {bad} -- lihat log di atas, jangan lanjut ke eksperimen utama."
    print("\nUji singkat OK. Untuk sesi berikutnya set RUN_SMOKE = False.")

## 5. Jalankan antrean eksperimen utama

In [ ]:
todo = [k for k in PLAN if not is_done(k)]
print(f"Selesai: {110 - len(todo)} / 110 | tersisa: {len(todo)}")
print("Berikutnya:", todo[:6])

deadline = NOTEBOOK_START + SESSION_BUDGET_H * 3600
finished, failed = run_queue(todo, deadline)
print(f"\nSesi ini: {len(finished)} run selesai, {len(failed)} gagal: {failed}")

## 6. Progres dan ringkasan sementara

F1 periode aktif dihitung dari `scores_test.npz` (periode ekor = mulai 11 Sep 2022 = detik ke-864.000 sejak transaksi pertama 1 Sep 2022 00:00, untuk HI dan LI).

In [ ]:
TAIL_START = 864000


def f1_at_half(y, s):
    pred = s > 0.5
    tp, fp, fn = (pred & (y == 1)).sum(), (pred & (y == 0)).sum(), (~pred & (y == 1)).sum()
    p = tp / (tp + fp) if tp + fp else 0.0
    r = tp / (tp + fn) if tp + fn else 0.0
    return 100 * (2 * p * r / (p + r) if p + r else 0.0)


rows = []
for key in PLAN:
    rd = run_dir_of(key)
    if not os.path.exists(f"{rd}/run_summary.json"):
        continue
    s = json.load(open(f"{rd}/run_summary.json"))
    d = np.load(f"{rd}/scores_test.npz")
    act = d["timestamp"] < TAIL_START
    rows.append({"dataset": key[0], "model": key[1], "config": key[2], "seed": key[3],
                 "epochs_run": s["epochs_run"], "best_epoch": s["best_epoch"],
                 "test_f1": 100 * s["test"]["f1"],
                 "f1_aktif": f1_at_half(d["y"][act].astype(int), d["score"][act]),
                 "lambda_per_day": s.get("temporal", {}).get("lambda_per_day"),
                 "run_jam": (sum(s["train_epoch_s"]) + sum(s["val_eval_s"]) + s["test_eval_s"]) / 3600})

runs = pd.DataFrame(rows)
runs.to_csv(f"{OUT_DIR}/progress_runs.csv", index=False)
print(f"Run selesai (termasuk dari sesi sebelumnya di Output ini): {len(runs)} / 110")
if len(runs):
    agg = runs.groupby(["dataset", "model", "config"]).agg(
        seeds=("seed", "count"), test_f1_mean=("test_f1", "mean"), test_f1_std=("test_f1", "std"),
        f1_aktif_mean=("f1_aktif", "mean"), f1_aktif_std=("f1_aktif", "std"),
        epochs_mean=("epochs_run", "mean"), jam_mean=("run_jam", "mean"))
    display(agg.round(2))

In [ ]:
!cd /kaggle/working && rm -f results_main.zip && zip -qr results_main.zip results_main -x "*/ckpt/*" && ls -lh results_main.zip
print(f"Total waktu notebook: {(time.time() - NOTEBOOK_START) / 3600:.2f} jam")